# Optuna HPO Analysis

Seeds an in-memory Optuna study from a Weights & Biases project and visualises
hyperparameter importance (fANOVA) and slice plots.

**Requirements:** run with the root workspace environment (`uv run jupyter notebook` from the repo root).

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
WANDB_PROJECT = "fed-dcn-synthetic-hpo-non-iid-mnist-accuracy"  # W&B project to load runs from
WANDB_ENTITY  = None          # W&B entity/username; None → currently logged-in user
OBJECTIVE     = "accuracy"    # "accuracy" (maximize) or "db_latent" (minimize)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import optuna
import wandb
from optuna.distributions import CategoricalDistribution, FloatDistribution, IntDistribution

optuna.logging.set_verbosity(optuna.logging.WARNING)

## Search space

Must match `experiment_runner._suggest_params` and `seed_study_from_wandb._DISTRIBUTIONS`.

In [ ]:
HIDDEN_DIMS_CHOICES = [
    "[512, 256, 128]",
    "[256, 128]",
    "[512, 256]",
    "[256, 128, 64]",
]

DISTRIBUTIONS = {
    "hidden_dims":          CategoricalDistribution(HIDDEN_DIMS_CHOICES),
    "bottleneck_dim":       IntDistribution(3, 16),
    "alpha_geom":           FloatDistribution(1e-4, 1.0, log=True),
    "alpha_geom_federated": FloatDistribution(1e-4, 1.0, log=True),
    "clust_weight":         FloatDistribution(1e-4, 1.0, log=True),
    "recon_weight":         FloatDistribution(0.1, 10.0, log=True),
    "learning_rate":        FloatDistribution(1e-5, 1e-2, log=True),
}

# Maps W&B config key (Flower naming) → Optuna parameter name
WANDB_KEY_TO_OPTUNA = {
    "hidden-dims":               "hidden_dims",
    "bottleneck-dim":            "bottleneck_dim",
    "alpha-geom":                "alpha_geom",
    "alpha-geom-federated":      "alpha_geom_federated",
    "lambda-clust-loss":         "clust_weight",
    "alpha-reconstruction-loss": "recon_weight",
    "learning-rate":             "learning_rate",
}

METRIC_KEY = {"accuracy": "train_acc", "db_latent": "train_db_latent"}[OBJECTIVE]

## Load runs from W&B and seed Optuna study

In [ ]:
def _normalize(v):
    if not isinstance(v, str):
        return v
    if v.lower() == "true":  return True
    if v.lower() == "false": return False
    try: return int(v)
    except ValueError: pass
    try: return float(v)
    except ValueError: pass
    return v


def _extract_params(run_config):
    params = {}
    for wandb_key, optuna_name in WANDB_KEY_TO_OPTUNA.items():
        raw = run_config.get(wandb_key)
        if raw is None:
            return None
        params[optuna_name] = _normalize(raw)
    return params


def _in_bounds(params):
    for name, dist in DISTRIBUTIONS.items():
        v = params[name]
        if isinstance(dist, CategoricalDistribution) and v not in dist.choices:
            return False
        if isinstance(dist, IntDistribution) and not (dist.low <= int(v) <= dist.high):
            return False
        if isinstance(dist, FloatDistribution) and not (dist.low <= float(v) <= dist.high):
            return False
    return True


api  = wandb.Api()
path = f"{WANDB_ENTITY}/{WANDB_PROJECT}" if WANDB_ENTITY else WANDB_PROJECT

print(f"Querying W&B project '{path}'...")
try:
    runs = list(api.runs(path, filters={"tags": {"$nin": ["repeat_run"]}}))
except (wandb.errors.CommError, ValueError) as e:
    raise RuntimeError(f"Could not load W&B project '{path}': {e}")

print(f"Found {len(runs)} run(s).")

direction = "maximize" if OBJECTIVE == "accuracy" else "minimize"
study = optuna.create_study(direction=direction)  # in-memory, no persistence

seeded = skipped = 0
for run in runs:
    obj_val = run.summary.get(METRIC_KEY)
    if obj_val is None:
        skipped += 1
        continue
    params = _extract_params(run.config)
    if params is None or not _in_bounds(params):
        skipped += 1
        continue
    params["bottleneck_dim"]       = int(params["bottleneck_dim"])
    params["alpha_geom"]           = float(params["alpha_geom"])
    params["alpha_geom_federated"] = float(params["alpha_geom_federated"])
    params["clust_weight"]         = float(params["clust_weight"])
    params["recon_weight"]         = float(params["recon_weight"])
    params["learning_rate"]        = float(params["learning_rate"])
    trial = optuna.trial.create_trial(
        params=params,
        distributions=DISTRIBUTIONS,
        value=float(obj_val),
    )
    study.add_trial(trial)
    seeded += 1

print(f"Seeded {seeded} trial(s), skipped {skipped}.")
print(f"Best trial: {METRIC_KEY} = {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

## Optimization history

In [ ]:
from optuna.visualization import plot_optimization_history

plot_optimization_history(study).show()

## Hyperparameter importance (fANOVA)

Requires ≥ 4 completed trials.

In [ ]:
from optuna.visualization import plot_param_importances

if len(study.trials) < 4:
    print(f"Only {len(study.trials)} trial(s) — need at least 4 for fANOVA importance.")
else:
    plot_param_importances(study).show()

## Slice plots

Marginal effect of each hyperparameter on the objective.

In [ ]:
from optuna.visualization import plot_slice

plot_slice(study).show()

## Parallel coordinates

Each line is one trial; colour encodes the objective value.

In [ ]:
from optuna.visualization import plot_parallel_coordinate

plot_parallel_coordinate(study).show()

## Contour plots (pairwise interactions)

In [ ]:
from optuna.visualization import plot_contour

# Focus on the continuous parameters most likely to interact
params_of_interest = ["alpha_geom", "alpha_geom_federated", "clust_weight", "recon_weight", "learning_rate"]

if len(study.trials) < 4:
    print(f"Only {len(study.trials)} trial(s) — need at least 4 for contour plots.")
else:
    plot_contour(study, params=params_of_interest).show()